In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-kaizen

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT09 — Exercise 1: LLM Architecture and Tokenization
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Understand decoder-only transformer internals — tokenization,
#   KV cache, parameter counting — and make your first LLM call via Delegate.
#
# TASKS:
#   1. Implement BPE tokenizer from scratch
#   2. Calculate model parameter count from architecture spec
#   3. Estimate KV cache memory requirements
#   4. Make first Delegate call with cost budget
#   5. Compare tokenizer output vs model's built-in tokenizer
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import os
import re
from collections import Counter

import polars as pl

from kaizen_agents import Delegate

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

if not os.environ.get("OPENAI_API_KEY"):
    print("\u26a0 OPENAI_API_KEY not set \u2014 skipping LLM exercises.")
    print("  Set it in .env to run this exercise with real LLM calls.")
    import sys

    sys.exit(0)

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
print(f"LLM Model: {model}")



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
reports = loader.load("ascent09", "sg_company_reports.parquet")

sample_texts = reports.select("text").head(5).to_series().to_list()
sample_text = (
    sample_texts[0] if sample_texts else "Singapore is a global financial hub."
)

print(f"Loaded {reports.height:,} documents")
print(f"Sample text (first 200 chars): {sample_text[:200]}...")



## TASK 1: Implement BPE tokenizer from scratch


In [ ]:
def get_word_freqs(text: str) -> dict[str, int]:
    """Split text into words and count frequencies."""
    words = re.findall(r"\w+|[^\w\s]", text.lower())
    return Counter(words)


def get_pair_freqs(vocab: dict[str, int]) -> Counter:
    """Count frequencies of adjacent character pairs across the vocabulary."""
    pairs = Counter()
    for word, freq in vocab.items():
        chars = list(word)
        for i in range(len(chars) - 1):
            pairs[(chars[i], chars[i + 1])] += freq
    return pairs


def train_bpe(
    text: str, num_merges: int = 50
) -> tuple[list[tuple[str, str]], dict[str, int]]:
    """Train BPE tokenizer: word freqs → char-level vocab → iterative pair merging.
    Each merge step: find most-frequent adjacent pair, record it, collapse it in vocab.
    Return (merges_list, final_vocab).
    """
    # TODO: Implement BPE training — word freqs, char-level vocab, merge loop.
    # Build the initial vocab by splitting each word into space-separated chars
    # (e.g., "hi" → "h i"). Each merge step collapses the most-common pair
    # ("h i" → "hi") across all vocab entries.
    ____
    ____
    ____


def tokenize_bpe(text: str, merges: list[tuple[str, str]]) -> list[str]:
    """Apply learned BPE merges to tokenize text.
    Split into words, then for each word apply merges left-to-right in order.
    """
    # TODO: For each word, start with individual chars, then apply each merge
    # rule in sequence: scan left-to-right, merge matching adjacent pairs in-place.
    words = re.findall(r"\w+|[^\w\s]", text.lower())
    tokens = []
    for word in words:
        ____
        ____
    return tokens


merges, vocab = train_bpe(sample_text, num_merges=50)
bpe_tokens = tokenize_bpe(sample_text[:500], merges)

print(f"\n=== BPE Tokenizer ===")
print(f"Learned {len(merges)} merge rules")
print(f"Top 10 merges: {merges[:10]}")
print(f"Vocab size: {len(vocab)}")
print(f"Sample tokenization ({len(bpe_tokens)} tokens): {bpe_tokens[:30]}...")



## TASK 2: Calculate model parameter count from architecture spec


In [ ]:
def count_parameters(
    vocab_size: int,
    d_model: int,
    n_layers: int,
    n_heads: int,
    d_ff: int,
) -> dict[str, int]:
    """Return a breakdown dict of parameter counts for a decoder-only transformer.
    Components: token_embedding, position_embedding (seq_len=4096),
    attention_per_layer (QKV+out), ffn_per_layer (up+down), layernorm_per_layer,
    total_transformer_layers, final_layernorm, lm_head, total_parameters.
    """
    max_seq_len = 4096
    # TODO: Compute each component and sum into total_parameters.
    # Attention per layer = Q+K+V projections + output projection (4 × d_model²).
    # FFN per layer = up-projection + down-projection (2 × d_model × d_ff).
    # Layer norm per layer = 2 norms × (weight + bias) each of size d_model.
    ____
    ____
    ____
    ____
    ____
    return {
        "token_embedding": token_emb,
        "position_embedding": pos_emb,
        "attention_per_layer": attn_per_layer,
        "ffn_per_layer": ffn_per_layer,
        "layernorm_per_layer": ln_per_layer,
        "total_transformer_layers": all_layers,
        "final_layernorm": final_ln,
        "lm_head": lm_head,
        "total_parameters": total,
    }


params = count_parameters(
    vocab_size=50_257, d_model=768, n_layers=12, n_heads=12, d_ff=3072
)
print(f"\n=== Parameter Count (GPT-2 Small equiv) ===")
for component, count in params.items():
    print(f"  {component}: {count:>15,}")
print(f"  Total: {params['total_parameters'] / 1e6:.1f}M parameters")



## TASK 3: Estimate KV cache memory requirements


In [ ]:
def estimate_kv_cache(
    batch_size: int,
    seq_len: int,
    n_layers: int,
    n_heads: int,
    d_head: int,
    dtype_bytes: int = 2,
) -> dict[str, float]:
    """Return bytes_per_token, total_bytes, total_mb, total_gb for a KV cache.
    Formula: per_token = 2 (K+V) × n_layers × n_heads × d_head × dtype_bytes.
    Total = batch_size × seq_len × per_token.
    """
    # TODO: Compute per_token bytes, total_bytes, convert to MB and GB.
    ____
    ____
    ____
    return {
        "bytes_per_token": per_token,
        "total_bytes": total_bytes,
        "total_mb": total_bytes / (1024**2),
        "total_gb": total_gb,
        "batch_size": batch_size,
        "seq_len": seq_len,
    }


kv_cache = estimate_kv_cache(
    batch_size=1, seq_len=4096, n_layers=32, n_heads=32, d_head=128, dtype_bytes=2
)
print(f"\n=== KV Cache Memory (7B model, seq_len=4096) ===")
print(f"  Bytes per token: {kv_cache['bytes_per_token']:,}")
print(f"  Total memory: {kv_cache['total_mb']:.1f} MB ({kv_cache['total_gb']:.3f} GB)")

print(f"\n  KV Cache Scaling:")
for seq in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
    kv = estimate_kv_cache(1, seq, 32, 32, 128, 2)
    print(f"    seq_len={seq:>6}: {kv['total_mb']:>8.1f} MB")



## TASK 4: Make first Delegate call with cost budget


In [ ]:
async def first_delegate_call():
    """Make a simple Delegate call to analyze document text."""
    # TODO: Create a Delegate with model= and budget_usd=1.0, then stream its
    # response to the prompt below, accumulating event.text into response_text.
    # The Delegate's budget cap is MANDATORY — always enforce cost limits.
    ____
    ____

    prompt = f"""Analyze this Singapore company report excerpt and identify:
1. Key financial metrics mentioned
2. Industry sector
3. Overall sentiment (positive/negative/neutral)

Text: {sample_text[:1000]}"""

    print(f"\n=== First Delegate Call ===")
    print(f"Prompt length: {len(prompt)} characters")
    ____
    print(f"Response ({len(response_text)} chars):")
    print(response_text[:500])
    return response_text


delegate_response  = await first_delegate_call()


## TASK 5: Compare tokenizer output vs model's built-in tokenizer


In [ ]:
async def compare_tokenizers():
    """Ask the Delegate to tokenize the same text and compare with our BPE."""
    # TODO: Create a Delegate (budget_usd=0.5). Tokenize test_sentence with our
    # BPE, then ask the model how it would tokenize the same sentence.
    # Stream and accumulate the response, then print both tokenizations.
    test_sentence = "Singapore's financial sector grew 8.2% in Q3."
    our_tokens = tokenize_bpe(test_sentence, merges)
    ____
    ____
    print(f"\n=== Tokenizer Comparison ===")
    print(f"Our BPE ({len(our_tokens)} tokens): {our_tokens}")
    print(f"Model's perspective: {response_text[:400]}")
    print(f"  - BPE principle is the same — merge frequent pairs")
    print(f"  - More training data = better subword boundaries")


await compare_tokenizers()
print(
    "\n✓ Exercise 1 complete — LLM architecture, tokenization, and first Delegate call"
)

